In [ ]:
pip install pandas matplotlib scipy numpy seaborn natsort

In [2]:
import copy
import pandas as pd
import math
import seaborn as sns
import itertools
import warnings
import requests
import subprocess
import numpy as np
from matplotlib import pyplot as plt
import matplotlib as mpl
from scipy import stats
from scipy.optimize import curve_fit
import json
import os
import sys
from collections import Counter
from scipy.stats import pearsonr
from IPython.display import display
class StopExecution(Exception):
    def _render_traceback_(self):
        pass
sys.path.insert(0, "../src")
from perf_tools.analysis import make_differential_frame, get_data, get_summary_statistics
from perf_tools.analysis import check_are_close, make_latency_plot, plot_latency_stats, MetricData

In [ ]:
VARIANTS = {"replset": "linux-1-node-replSet-fle.2022-11"}

WORKDIR="../datasets/genny-range/experiment_q3"
#WORKDIR="../datasets/genny-range/experiment_0"

patch_ids = ['6566270b30661516d325690c']

In [ ]:

EXPERIMENTS = [
  # Experiment Set 0: fix
  # {
  # "name": "es0",
  # "coll": "OneFieldOneValue",
  # "encryptedFieldCount" : 1,
  # "upperBounds" : [2**9-1, 2**31-1],
  # "contentionFactors" : [0,8],
  # "sparsities": [1,4],
  # "bigsmall": [False, True],
  # "queryThreads": 1,
  # },
  # Experiment Set 1.1: blah
  # {
  # "name": "es1",
  # "coll": "OneFieldOneValue",
  # "encryptedFieldCount" : 1,
  # "upperBounds" : [2**9-1, 2**13-1, 2**17-1, 2**31-1],
  # "contentionFactors" : [0,4,8],
  # "sparsities": [1,2,3,4],
  # "bigsmall": [False],
  # "queryThreads": 1,
  # },
  # Experiment Set 2.encrypted: 
  # {
  # "name": "es2_encrypted",
  # "coll": "ContiguousValuesFreqOne",
  # "encryptedFieldCount" : 1,
  # "diff": {
  #   "field": [
  #       {'name': 'f_sint32_1'},
  #       {'name': 'f_sint32_2'},
  #       {'name': 'f_bin64_1'},
  #       {'name': 'f_bin64_2'},
  #       {'name': 'f_dec128_1'},
  #       {'name': 'f_dec128_2'},
  #   ],
  #   "selectivity" : [5, 100, 1000, 10000],
  #   "contentionFactor" : [0,4,8],
  #   "sparsity": [1,2,3,4],
  #   "queryPattern": ["fixed", "rand"],
  # },
  # "queryCount": 10000,
  # "queryThreads": 1,
  # },
  # Experiment Set 2.unencrypted:
  # {
  # "name": "es2_unencrypted",
  # "coll": "ContiguousValuesFreqOne",
  # "encryptedFieldCount" : 0,
  # "diff": {
  #   "field": [
  #       {'name': 'f_sint32_1'},
  #       {'name': 'f_sint32_2'},
  #       {'name': 'f_bin64_1'},
  #       {'name': 'f_bin64_2'},
  #       {'name': 'f_dec128_1'},
  #       {'name': 'f_dec128_2'},
  #   ],
  #   "selectivity" : [5, 100, 1000, 10000],
  #   "queryPattern": ["fixed", "rand"],
  # },
  # "queryCount": 10000,
  # "queryThreads": 1,
  # }
  # Experiment set Q3
  {
  "name": "experiment_q3",
  "coll": "OneFieldSmallSupport",
  "encryptedFieldCount" : 1,
  "diff": {
    "sel" : [10, 100, 1000, 10000],
    "cf" : [0,4,8],
    "sp": [1,2,3,4],
    "pat": ["fixed", "rand"],
    "tf": [0, 2, 4, 6, 8]
  },
  "queryCount": 10000,
  "queryThreads": 1,
  },

]

In [ ]:
# EXPERIMENT 1

# class DataSetCache:
#     def __init__(self, workdir, patch_id, variant, execution, task_name, experiment, cf, sp, ub, isbig, range, metric_name):
#         self.workdir = workdir
#         self.patch_id = patch_id
#         self.variant = variant
#         self.execution = execution
#         self.task_name = task_name
#         self.experiment = experiment
#         self.cf = cf
#         self.sp = sp
#         self.ub = ub
#         self.isbig = isbig
#         self.range = range
#         self.metric_name = metric_name
#         self.variant = variant

#         self.data = None

#     def json_path(self, metric):
#         return os.path.join(self.workdir, self.patch_id, self.variant,
#             self.task_name, str(self.execution), metric + ".json")

#     def get_data(self):
#         if self.data is None:
#             self.data = get_data(self.json_path(self.metric_name))
#         return self.data

# DATASETS = {}

# for ex in EXPERIMENTS:
#     for cf in ex["contentionFactors"]:
#         for sp in ex["sparsities"]:
#             for ub in ex["upperBounds"]:
#                 for isBig in ex["bigsmall"]:
#                     experiment = ex['name']
#                     testName = f"qe_range_testing_workloads_evergreen_experiment1_1_c{cf}_s{sp}_ub{ub}"
#                     #testName = f"qe_range_testing_workloads_evergreen_experiment0_c{cf}_s{sp}_ub{ub}_{'big' if isBig else 'small'}"
#                     fullName = testName + ".query"

#                     #DATASETS[fullName] = DataSetCache(WORKDIR, patch_id, VARIANTS["replset"], "0", testName, experiment, cf, tc, "load", "load", "load.inserts")
#                     caches = []
#                     for i in range(ex["queryThreads"]): # threads
#                         ds = DataSetCache(WORKDIR, patch_id, VARIANTS["replset"], "0", testName, experiment, cf, sp, ub, isBig, (i * 100000//ex["queryThreads"], (i+1) * 100000//ex["queryThreads"]), f"PreDefinedRangeQueryActor_Thread{i}.query.range_query")
#                         try:
#                             with open(ds.json_path(ds.metric_name), 'r') as f:
#                                 pass
#                         except:
#                             #todo
#                             ds = DataSetCache(WORKDIR, patch_id, VARIANTS["replset"], "1", testName, experiment, cf, sp, ub, isBig, (i * 100000//ex["queryThreads"], (i+1) * 100000//ex["queryThreads"]), f"PreDefinedRangeQueryActor_Thread{i}.query.range_query")
#                             try:
#                                 with open(ds.json_path(ds.metric_name), 'r') as f:
#                                     pass
#                             except Exception as e:
#                                 print('Nothing at '+ ds.json_path(ds.metric_name))
#                                 continue
#                         caches.append(ds)
#                     if(len(caches) == 0):
#                         print('Failed to load ' + testName)
#                     else: 
#                         DATASETS[fullName] = caches

In [ ]:
# EXPERIMENT 2
    
class Varset:
    def __getattr__(self, __name):
        return self.varset_dict[__name]

class Experiment2EncryptedVarset(Varset):
    def __init__(self, varset_dict):
        self.varset_dict = varset_dict

    def get_name(self):
        name = self.varset_dict['field']['name']
        cf = self.varset_dict['contentionFactor']
        sp = self.varset_dict['sparsity']
        sel = self.varset_dict['selectivity']
        pat = self.varset_dict['queryPattern']
        return f'experiment2_encrypted_{name}_sp{sp}_cf{cf}_sel{sel}_{pat}'
    
    def get_metric_name(self):
        name = self.varset_dict['field']['name']
        sel = self.varset_dict['selectivity']
        pat = self.varset_dict['queryPattern']
        return f"PreDefinedRangeQueryActor.query.range_query_{name}_{pat}_sel{sel}"

class Experiment2UnencryptedVarset(Varset):
    def __init__(self, varset_dict):
        self.varset_dict = varset_dict

    def get_name(self):
        name = self.varset_dict['field']['name']
        sel = self.varset_dict['selectivity']
        pat = self.varset_dict['queryPattern']
        return f'experiment2_unencrypted_{name}_sel{sel}_{pat}'
    
    def get_metric_name(self):
        name = self.varset_dict['field']['name']
        sel = self.varset_dict['selectivity']
        pat = self.varset_dict['queryPattern']
        return f"PreDefinedRangeQueryActor.query.range_query_{name}_{pat}_sel{sel}"

In [ ]:
# EXPERIMENT 3

class Experiment3Varset(Varset):
    def __init__(self, varset_dict):
        self.varset_dict = varset_dict

    def get_name(self):
        cf = self.varset_dict['cf']
        sp = self.varset_dict['sp']
        sel = self.varset_dict['sel']
        pat = self.varset_dict['pat']
        tf = self.varset_dict['tf']
        return f'experiment_q3_{pat}_sel{sel}_c{cf}_s{sp}_tf{tf}'
    
    def get_metric_name(self):
        return f"PreDefinedRangeQueryActor.query.range_query"

In [ ]:
VARSET_MAP = {
    'es2_encrypted': Experiment2EncryptedVarset, 
    'es2_unencrypted': Experiment2UnencryptedVarset,
    'experiment_q3': Experiment3Varset
}

In [ ]:
    
                            
class DataSetCache:
    def __init__(self, experiment_name, workdir, patch_id, variant, varset):
        self.experiment_name = experiment_name
        self.workdir = workdir
        self.patch_id = patch_id
        self.variant = variant
        self.varset = varset
        self.metric_name = varset.get_metric_name()
        self.task_name = varset.get_name()

        self.data = None
        self.execution = 0

    
    def json_path(self, metric):
        return os.path.join(self.workdir, self.patch_id, self.variant,
            'qe_range_testing_workloads_evergreen_' + self.task_name, str(self.execution), metric + ".json")

    def get_data(self):
        if self.data is None:
            self.data = get_data(self.json_path(self.metric_name))
        return self.data

    def try_find_data(self):
        self.execution = 0
        try:
            print(f'Trying {ds.json_path(ds.metric_name)}')
            with open(ds.json_path(ds.metric_name), 'r') as f:
                return True
        except:
            self.execution += 1
            try:
                print(f'Trying {ds.json_path(ds.metric_name)}')
                with open(ds.json_path(ds.metric_name), 'r') as f:
                    return True
            except:
                return False

DATASETS = []

def varset_iter(varset_class, diff):
    keys = list(diff.keys())
    return (varset_class({keys[i]: p[i] for i in range(len(keys))}) for p in itertools.product(*[diff[key] for key in diff]))

for patch_id in patch_ids:
    for ex in EXPERIMENTS:
        for varset in varset_iter(VARSET_MAP[ex['name']], ex['diff']):
            ds = DataSetCache(ex['name'], WORKDIR, patch_id, VARIANTS["replset"], varset)
            if ds.try_find_data():
                DATASETS.append(ds)

print(f'Total number of datasets found: {len(DATASETS)}')


In [ ]:
enc_diff_datas = []
unenc_diff_datas = []
diff_data = {}

for ds in DATASETS:
    ex_name = ds.experiment_name
    data = ds.get_data()
    # We are only interested in "fixed data"; trim first entry because it's often garbage
    fixed = data.fixed_data.iloc[1:]
    if ex_name not in diff_data:
        diff_data[ex_name] = []
    diff_data[ex_name].append((ds, fixed))
    

In [ ]:

# mcov_dir = '/home/genny/src/workloads/contrib/qe_range_testing/queries/'
# def mcov_file(ub, sp):
#     return mcov_dir + f'experiment2_coversize_ub{ub}_sp{sp}.txt'

# min_cover_sizes = {}

# for ex in EXPERIMENTS:
#     for ub in ex["upperBounds"]:
#         for sp in ex["sparsities"]:
#             with open(mcov_file(ub, sp)) as f:
#                 mcovsizes = [int(s.strip()) for s in f.readlines()]
#             # Since we are stripping the first query from each thread, we must remove the matching coversizes.
#             del mcovsizes[0::100000//ex["queryThreads"]]
#             min_cover_sizes[(ub, sp)] = mcovsizes

# for key, mc in min_cover_sizes.items():
#     ub, sp = key
#     print(ub, sp, Counter(mc))

In [ ]:
# for name, t in diff_datas.items():
#     caches, data = t
#     ub = caches[0].ub
#     sp = caches[0].sp
#     mincovers = min_cover_sizes[(ub, sp)]
#     data['min_cover_size'] = mincovers
#     print(data)
# print(diff_datas)

In [ ]:
# # Join data for experiment 2
# for ds, data in enc_diff_datas:
#     sp = ds.varset.varset_dict['sparsity']
#     cf = ds.varset.varset_dict['contentionFactor']
#     name = ds.varset.varset_dict['field']["name"]
#     sel = ds.varset.varset_dict['selectivity']
#     pat = ds.varset.varset_dict['queryPattern']

#     data['field_name'] = name
#     data['sp'] = sp
#     data['cf'] = cf
#     data['sel'] = sel
#     data['pat'] = pat

# for ds, data in unenc_diff_datas:
#     name = ds.varset.varset_dict['field']["name"]
#     sel = ds.varset.varset_dict['selectivity']
#     pat = ds.varset.varset_dict['queryPattern']

#     data['field_name'] = name
#     data['sel'] = sel
#     data['pat'] = pat

# all_data = pd.concat(data for _, data in enc_diff_datas)
# all_data.reset_index(inplace=True)
# all_data

# all_unenc_data = pd.concat(data for _, data in unenc_diff_datas)
# all_unenc_data.reset_index(inplace=True)
# all_unenc_data

# field_to_size_map = {'f_sint32_1': 2**32, 'f_sint32_2': 100000, 'f_bin64_1': 2**64, 'f_bin64_2': 2000000, 'f_dec128_1': 2**128, 'f_dec128_2': 1000000000000}
# all_data['domain_size'] = all_data['field_name'].apply(lambda f: field_to_size_map[f])
# all_data['log_domain_size'] = all_data['domain_size'].apply(lambda f: math.log2(f))
# all_unenc_data['domain_size'] = all_unenc_data['field_name'].apply(lambda f: field_to_size_map[f])
# all_unenc_data['log_domain_size'] = all_unenc_data['domain_size'].apply(lambda f: math.log2(f))

# all_data.to_pickle("all_data.pkl")
# all_unenc_data.to_pickle("all_unenc_data.pkl")

In [ ]:
# Join data for experiment Q3
for ex, dd in diff_data.items():
    for ds, data in dd:
        sp = ds.varset.varset_dict['sp']
        cf = ds.varset.varset_dict['cf']
        sel = ds.varset.varset_dict['sel']
        pat = ds.varset.varset_dict['pat']
        tf = ds.varset.varset_dict['tf']

        data['tf'] = tf
        data['sp'] = sp
        data['cf'] = cf
        data['sel'] = sel
        data['pat'] = pat
        data['ex_name'] = ex

all_data = pd.concat(data for diffdat in diff_data.values() for _, data in diffdat)
all_data.reset_index(inplace=True)
all_data

all_data['latency'] = all_data['d(t_pure)'] / 10**6

# fixed pattern data should not have been tested, oops!
all_data = all_data[all_data['pat'] == 'rand']
all_data.reset_index(inplace=True)

all_data.to_pickle("all_data.pkl")

In [3]:
all_data = pd.read_pickle('all_data.pkl')

In [ ]:
# all_data[all_data['sel'] == 1000].groupby(by='tf').latency.mean().plot()
# plt.show()
# plt.close()

In [4]:
# Exp Q3 plot 1
# blues = itertools.cycle(iter(mpl.cycler('color', [plt.get_cmap('Blues')(1-1. * i/6) for i in range(5)])))

for cf in [0, 4, 8]:
    for sp in [1, 2, 3, 4]:
        data = all_data[(all_data['cf'] == cf) & (all_data['sp'] == sp)]
        data.groupby(by=['sel', 'tf']).latency.mean().unstack(level=1).plot.bar()
        plt.title(f'Average latency vs query size + trimming\nsparsity={sp}, contention={cf}')
        plt.ylabel('Average latency (ms)')
        plt.xlabel('Query size')
        plt.legend(title='Trim factor')
        # plt.show()
        plt.tight_layout()
        plt.savefig(f'figures/q3_barplot_vs_size_trimming_sp{sp}_cf{cf}.png')
        plt.close()


In [24]:
# Exp Q3 plot 2
# blues = itertools.cycle(iter(mpl.cycler('color', [plt.get_cmap('Blues')(1-1. * i/6) for i in range(5)])))
main_group_contention = True
unstack_level = 1 if main_group_contention else 0
for sel in [10, 100, 1000, 10000]:
    for sp in [1, 2, 3, 4]:
        data = all_data[(all_data['sel'] == sel) & (all_data['sp'] == sp)]
        data.groupby(by=['cf', 'tf']).latency.mean().unstack(level=unstack_level).plot.bar()
        plt.title(f'Average latency vs trimming + contention\nsparsity={sp}, query size={sel}')
        plt.ylabel('Average latency (ms)')
        plt.xlabel('Contention factor' if main_group_contention else 'Trim factor')
        plt.legend(title='Trim factor' if main_group_contention else 'Contention')
        plt.tight_layout()
        # plt.show()
        plt.savefig(f'figures/q3_barplot_vs_trimming_contention_swapped_grouping_size{sel}_sp{sp}.png')
        plt.close()


In [22]:
# Exp Q3 plot 3 with the inserts
insert_data = pd.read_pickle('q3_summed_throughput.pkl')
insert_data
main_group_contention = False
unstack_level = 0 if main_group_contention else 1
for sel in [10, 100, 1000, 10000]:
    for sp in [1, 2, 3, 4]:
        q_data = all_data[(all_data['sel'] == sel) & (all_data['sp'] == sp)]
        i_data = insert_data[(insert_data['sparsity'] == sp)]
        fig, ax1 = plt.subplots()
        # NOTE unstack level = 0 to generate first graphs, 1 to generate second
        q_data.groupby(by=['tf', 'cf']).latency.mean().unstack(level=unstack_level).plot.bar(ax=ax1)
        
        print(ax1.get_ylim())
        
        ax2 = ax1.twinx()
        
        ((i_data.groupby(by=['trim_factor', 'contention']).OperationThroughput.mean().unstack(level=unstack_level))*-1).plot.bar(ax=ax2)
        print(ax2.get_ylim())
        
        ax1.set_ylim(bottom=-ax1.get_ylim()[1])
        ax2.set_ylim(top=-ax2.get_ylim()[0])
        top_ticks = sorted(list(i for i in ax1.get_yticks() if i > 0))
        bot_ticks = sorted(list(i for i in ax2.get_yticks() if i > 0))
        ax1.set_ylim(bottom=-ax1.get_ylim()[1])
        ax2.set_ylim(top=-ax2.get_ylim()[0])
        print(top_ticks, bot_ticks)
        ax2_realticks = [i for i in ax2.get_yticks() if i < 0]
    
        ax2.set_yticks(ticks=ax2_realticks, labels=[-i for i in ax2_realticks])
        ax1.set_yticks([i for i in ax1.get_yticks() if i >= 0])
        ax1.set_ylim(bottom=-ax1.get_ylim()[1])
        ax2.set_ylim(top=-ax2.get_ylim()[0])
        ax2.yaxis.tick_left()
        ax2.yaxis.set_label_position('left')
        # ax2.set_yticks([0,1,8])
        # ax1.set_yticks([0,1,8])
        #bot_ticks[::-1] + [0] + top_ticks)
        ax2.set_ylabel('  Insert throughput (docs/s)     Average query latency (ms)')
        plt.axhline(0, color='gray', linestyle='--')
        
        ax2.get_legend().remove()
        ax1.get_legend().remove()
        plt.legend(title='Trim factor' if main_group_contention else 'Contention', loc='upper left')
        #plt.legend()
        plt.title(f'Insert throughput + query latency comparison\nvs contention & trim factor\nquery size={sel}, sparsity={sp}')
        ax1.set_xlabel('Contention factor' if main_group_contention else 'Trim factor')
        plt.tight_layout()

        # plt.savefig(f'figures/q3_insert_vs_query_trim_factor_groups_size{sel}_sp{sp}.png')
        plt.show()
        plt.close()

# fig, ax1 = plt.subplots()
# t = range(4)
# ax1.bar(t, [1,2,3,4])
# ax1.set_ylim(bottom=-4)
# ax2 = ax1.twinx()
# ax2.bar(t, [-1,-2,-10,-15])
# ax2.set_ylim(top=15)
# fig.tight_layout()
# plt.show()
# plt.close()



(0.0, 3.923205905175517)
(-910.1338641320154, 0.0)
[1.0, 2.0, 3.0, 4.0] [250.0, 500.0, 750.0, 1000.0]
(0.0, 4.608239070342035)
(-1002.2771126273935, 0.0)
[2.0, 4.0, 6.0] [250.0, 500.0, 750.0, 1000.0, 1250.0]
(0.0, 7.155180144659465)
(-1105.1865370128785, 0.0)
[2.0, 4.0, 6.0, 8.0] [250.0, 500.0, 750.0, 1000.0, 1250.0]
(0.0, 6.929131741254125)
(-1096.4597424194908, 0.0)
[2.0, 4.0, 6.0, 8.0] [250.0, 500.0, 750.0, 1000.0, 1250.0]
(0.0, 7.4218133959946)
(-910.1338641320154, 0.0)
[2.0, 4.0, 6.0, 8.0] [250.0, 500.0, 750.0, 1000.0]
(0.0, 9.141105111326132)
(-1002.2771126273935, 0.0)
[2.5, 5.0, 7.5, 10.0] [250.0, 500.0, 750.0, 1000.0, 1250.0]
(0.0, 13.939391651230123)
(-1105.1865370128785, 0.0)
[5.0, 10.0, 15.0] [250.0, 500.0, 750.0, 1000.0, 1250.0]
(0.0, 17.320288211266128)
(-1096.4597424194908, 0.0)
[5.0, 10.0, 15.0, 20.0] [250.0, 500.0, 750.0, 1000.0, 1250.0]
(0.0, 14.002039779627962)
(-910.1338641320154, 0.0)
[5.0, 10.0, 15.0] [250.0, 500.0, 750.0, 1000.0]
(0.0, 16.291577623357338)
(-1002.2

In [ ]:
# Various exp 2 plots

# Exp 2 average plots
# groupers = [('log_domain_size', 'log2(domain size)', (17, 21, 32, 40, 64, 128)), ('sp', 'sparsity', [1, 2, 3, 4]), ('cf', 'contention', [0, 4, 8]), ('sel', 'selectivity', [5, 1000, 10000]), ('pat', 'query pattern', None), ('field_name', 'field name', None)]
# for col, full_name, xticks in groupers:
#     g = (all_data.groupby(by=col)['d(t_pure)'].mean() / 10**6)
#     print(g)
#     if xticks:
#         g.plot(style='o-')
#         plt.ylim(bottom=0)
#         plt.ylim(top=plt.ylim()[1]*1.1)
#         plt.title(f'Average query latency vs {full_name}, averaged over all experiments')
#         plt.ylabel('Average query latency (ms)')
#         plt.xlabel(full_name)
#         plt.xticks(xticks)
#         plt.show()
#         plt.close()
    
# # Exp 2 logplots
# groupers = [('sel', 'selectivity', [5, 1000, 10000])]
# for col, full_name, xticks in groupers:
#     if xticks is not None:
#         xticks = pd.DataFrame(xticks).squeeze().astype(float)
#         print(xticks)
#     (all_data.groupby(by=col)['d(t_pure)'].mean() / 10**6).plot(logx = True, style='o-')
#     plt.ylim(bottom=0)
#     plt.ylim(top=plt.ylim()[1]*1.1)
#     plt.title(f'Average query latency vs log({full_name}), averaged over all experiments')
#     plt.ylabel('Average query latency (ms)')
#     plt.xlabel(full_name)
#     plt.xticks(ticks=xticks)
#     plt.gca().get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
#     plt.show()
#     plt.close()
    
# exp 2
# groupers = [('log_domain_size', 'log2(domain size)', (17, 21, 32, 40, 64, 128)), ('sp', 'sparsity', [1, 2, 3, 4]), ('cf', 'contention', [0, 4, 8]), ('pat', 'query pattern', None), ('field_name', 'field name', None)]
# for col, full_name, xticks in groupers:
#     g = (all_data.groupby(by=[col,'sel'])['d(t_pure)'].mean() / 10**6).unstack(level=1)
#     print(g)
#     for sel in [5, 100, 1000, 10000]:
#         tograph = g[sel]
#         if xticks:
#             tograph.plot(style='o-')
#             print(tograph)
#             plt.ylim(bottom=0)
#             plt.ylim(top=plt.ylim()[1]*1.1)
#             plt.title(f'Average query latency vs {full_name}, selectivity={sel}')
#             plt.ylabel('Average query latency (ms)')
#             plt.xlabel(full_name)
#             plt.xticks(xticks)
#             plt.show()
#             plt.close()
    

# exp 2
# groupers = [('sp', 'sparsity', [1, 2, 3, 4])]
# for col, full_name, xticks in groupers:
#     g = (all_data[all_data['pat']=='rand'].groupby(by=[col,'sel','field_name'])['d(t_pure)'].mean() / 10**6).unstack(level=[1, 2])
#     print(g)
#     for sel in [5, 100, 1000, 10000]:
#         for field in ['f_sint32_1', 'f_sint32_2', 'f_bin64_1', 'f_bin64_2', 'f_dec128_1', 'f_dec128_2']:
#             tograph = g[sel][field]
#             if xticks:
#                 tograph.plot(style='o-')
#                 print(tograph)
#                 plt.ylim(bottom=0)
#                 plt.ylim(top=plt.ylim()[1]*1.1)
#                 plt.title(f'Average query latency vs {full_name}, selectivity={sel}, field={field}')
#                 plt.ylabel('Average query latency (ms)')
#                 plt.xlabel(full_name)
#                 plt.xticks(xticks)
#                 plt.show()
#                 plt.close()
    
# bounded = all_data['field_name'].apply(lambda s: s[-1] == "2")
# all_data['bounded'] = bounded
# exp 2
# for cf in [0, 4, 8]:
#     for sp in [1, 2, 3, 4]:
#         for qtype in ['rand', 'fixed']:
#             dataset = all_data[(all_data['pat'] == qtype) & (all_data['sp'] == sp) & (all_data['cf'] == cf)]
#             unbounded = dataset[~dataset['bounded']]
#             bounded = dataset[dataset['bounded']]
#             # plot 1
#             unbounded_grouped = unbounded.groupby(by=['field_name', 'sel'])['d(t_pure)'].mean() / 10**6
#             unbounded_grouped.unstack(level=0).plot(logx = True, logy=True, style='o-')
#             plt.ylim(bottom=0)
#             plt.ylabel('Average query latency (ms)')
#             plt.xlabel('Selectivity')
#             plt.title(f'Average latency vs selectivity, unbounded fields\ncontention={cf}, sparsity={sp}, query type={qtype}')
#             plt.savefig(f'figures/latency_unbounded_cf{cf}_sp{sp}_{qtype}.png')
#             plt.close()
#             # plot 1
#             bounded_grouped = bounded.groupby(by=['field_name', 'sel'])['d(t_pure)'].mean() / 10**6
#             bounded_grouped.unstack(level=0).plot(logx = True, logy=True, style='o-')
#             plt.title(f'Average latency vs selectivity, bounded fields\ncontention={cf}, sparsity={sp}, query type={qtype}')
#             plt.ylim(bottom=0)
#             plt.ylabel('Average query latency (ms)')
#             plt.xlabel('Selectivity')
#             plt.savefig(f'figures/latency_bounded_cf{cf}_sp{sp}_{qtype}.png')
#             plt.close()

# TODO redo graph 2 :(

# for cf in [0, 4, 8]:
#     for sp in [1, 2, 3, 4]:
#         for fname_stem in ['f_sint32_', 'f_bin64_', 'f_dec128_']:
#             f1 = fname_stem + '1'
#             f2 = fname_stem + '2'
#             dataset = all_data[(all_data['sp'] == sp) & (all_data['cf'] == cf) & ((all_data['field_name'] == f1) | (all_data['field_name'] == f2)) & all_data['sel'] <= 1000]
#             # plot 1
#             dataset['Query pattern'] = dataset['pat']
#             grouped = dataset.groupby(by=['field_name', 'sel'])['d(t_pure)'].mean() / 10**6
#             grouped.unstack(level=1).plot.bar()
#             plt.ylabel('Average query latency (ms)')
#             plt.xlabel('')
#             plt.title(f'Average latency vs field + query pattern\ncontention={cf}, sparsity={sp}, selectivity={sel}')
#             plt.show()
#             plt.close()
            

# for cf in [0, 4, 8]:
#     for sp in [1, 2, 3, 4]:
#         for fname_stem in ['f_sint32_', 'f_bin64_', 'f_dec128_']:
#             f1 = fname_stem + '1'
#             f2 = fname_stem + '2'
        
#             dataset = all_data[(all_data['pat'] == 'rand') & (all_data['sp'] == sp) & (all_data['cf'] == cf) & ((all_data['field_name'] == f1) | (all_data['field_name'] == f2))& (all_data['sel'] <= 1000)]

#             # plot 1
#             dataset['Query pattern'] = dataset['pat']
#             grouped = dataset.groupby(by=['field_name', 'sel'])['d(t_pure)'].mean() / 10**6
#             grouped.unstack(level=0).plot.bar()
#             plt.ylabel('Average query latency (ms)')
#             #plt.xlabe(['small', 'medium', 'large'])
#             plt.xlabel('Selectivity')
#             plt.title(f'Average latency vs field + selectivity\ncontention={cf}, sparsity={sp}, query type=rand')
#             plt.savefig(f'figures/latency_cf{cf}_sp{sp}_{fname_stem[:-1]}.png')
#             plt.close()
            
            

# # unenc vs enc
# for sel in [5, 100, 1000, 10000]:
#     unenc_dataset = all_unenc_data[all_unenc_data['sel'] == sel]
#     unenc_grouped = unenc_dataset.groupby(by='field_name')['d(t_pure)'].mean() / 10**6
#     unenc_grouped['encrypted'] = [False] * len(unenc_grouped)
#     for cf in [0, 4, 8]:
#         for sp in [1, 2, 3, 4]:
#             enc_dataset = all_data[(all_data['sel'] == sel) & (all_data['sp'] == sp) & (all_data['cf'] == cf)]
#             enc_grouped = enc_dataset.groupby(by='field_name')['d(t_pure)'].mean() / 10**6
#             enc_grouped['encrypted'] = [True] * len(unenc_grouped)
#             print(pd.concat((unenc_grouped, enc_grouped)))#.groupby(by='encrypted').bar.plot()
#             # plot 1
#             # dataset['Query pattern'] = dataset['pat']
#             # grouped = dataset.groupby(by=['field_name', 'sel'])['d(t_pure)'].mean() / 10**6
#             # grouped.unstack(level=0).plot.bar()
#             # plt.ylabel('Average query latency (ms)')
#             # #plt.xlabe(['small', 'medium', 'large'])
#             # plt.xlabel('Selectivity')
#             # plt.title(f'Average latency vs field + selectivity\ncontention={cf}, sparsity={sp}, query type=rand')
#             # plt.savefig(f'figures/latency_cf{cf}_sp{sp}_{fname_stem[:-1]}.png')
#             # plt.close()
            

In [ ]:
# Various analyses for experiment 1

# # Generate correlations
# pcorrsum = 0
# for name, t in diff_datas.items():
#     caches, data = t
#     ub = caches[0].ub
#     sp = caches[0].sp
#     cf = caches[0].cf
#     pcorr = pearsonr(data['min_cover_size'], (data['d(t_pure)']))
#     ocorr = data['min_cover_size'].corr(data['d(t_overhead)'])
#     tcorr = data['index'].corr(data['d(t_pure)'])
#     print(f"ub={ub},sp={sp},cf={cf},latencycorr={pcorr}, overheadcorr={ocorr}, timecorr={tcorr}")
#     #pcorrsum += pcorr
# pcorrsum /= len(diff_datas)
# print(f"Average correlation between min cover size and time of op: {pcorrsum}")
    
# # Histograms for latency
# for name, t in diff_datas.items():
#     caches, data = t
#     ub = caches[0].ub
#     sp = caches[0].sp
#     cf = caches[0].cf
#     size = 'big' if caches[0].isbig else 'small'
#     title = f"Experiment 1 upper bound={ub},sparsity={sp},contention={cf}, query latency"
#     data[title] = data["d(t_pure)"]
#     q = data[title].quantile(0.999)
#     hist = data[data[title] < q].hist(column=title, bins=1000)
#     #hist.title(f"Experiment 0 upper bound={ub},sparsity={sp},contention={cf},query size={size}")
#     hist
    
# # Density plots per cover size

# def color_lerp(c1, c2, f):
#     r1, g1, b1 = c1
#     r2, g2, b2 = c2
#     return (r2 * f + r1 * (1 - f), 
#             g2 * f + g1 * (1 - f), 
#             b2 * f + b1 * (1 - f)) 

# def color_lerp_list(c1, c2, n):
#     colors = []
#     for i in range(n-1):
#         factor = i * 1.0 / (n - 1)
#         colors.append(color_lerp(c1, c2, factor))
#     colors.append(c2)
#     return colors
# plt.ioff()
# for name, t in diff_datas.items():
#     caches, data = t
#     ub = caches[0].ub
#     sp = caches[0].sp
#     cf = caches[0].cf
#     title = f"Experiment 1 upper bound={ub},sparsity={sp},contention={cf}\nDensity of query latency (grouped by cover size)"
#     datacopy = data.copy(deep=True)
#     q = datacopy['d(t_pure)'].quantile(0.995)
#     datacopy = datacopy[datacopy['d(t_pure)'] < q]
#     for c in datacopy['min_cover_size'].unique():
#         if len(datacopy[datacopy['min_cover_size'] == c]) < 10:
#             print('Clipping ' + str(c))
#             datacopy = datacopy[datacopy['min_cover_size'] != c]
    
#     ax = sns.kdeplot(data=datacopy, x='d(t_pure)', hue='min_cover_size', fill=True, alpha=.5, linewidth=0, common_norm=False, palette='crest')
    
#     norm = mpl.colors.Normalize(vmin=datacopy['min_cover_size'].min(), vmax=datacopy['min_cover_size'].max())
#     sm = mpl.cm.ScalarMappable(norm=norm, cmap='crest')
#     sm.set_array([])
#     ax.get_legend().remove()
#     ax.figure.colorbar(sm, orientation='vertical', label='Min cover size')
#     plt.xlabel('Query Latency (ns)')
#     plt.title(title)
#     plt.savefig(f'figures/latency_density_ub{ub}_sp{sp}_cf{cf}.png')
#     plt.close()
#     #plt.show()
#     #d2.plot.density(color=color_lerp_list((0, 0, 1), (0, 1, 0), (len(d2.columns))))
#     #data.hist(column=title, by='min_cover_size')
    
# violinplots = []

# # Violin plots
# for name, t in diff_datas.items():
#     caches, data = t
#     ub = caches[0].ub
#     sp = caches[0].sp
#     cf = caches[0].cf
#     title = f"Experiment 1 upper bound={ub},sparsity={sp},contention={cf}, query latency"
#     tcol = 'd(t_pure)'
#     q = data[tcol].quantile(0.995)
#     data_cut = data[data[tcol] < q]
#     #data.boxplot(column=title, by="min_cover_size")
#     #data.groupby('min_cover_size')
#     #data[title].plot.kde()
#     factor = data_cut['min_cover_size'].nunique() // 15 + 1
#     data_cut['min_cover_size'] = data_cut['min_cover_size'].astype(int) // factor * factor

#     sns.violinplot(data_cut, x='min_cover_size', y=tcol, inner=None).set(title=title)
#     plt.show()


    
# # Line graph of 
# for name, t in diff_datas.items():
#     caches, data = t
#     ub = caches[0].ub
#     sp = caches[0].sp
#     cf = caches[0].cf
#     title = f"upper bound={ub},sparsity={sp},contention={cf}"
#     tcol = 'd(t_pure)'
#     # q = data[tcol].quantile(0.995)
#     # data_cut = data[data[tcol] < q]
#     data.boxplot(column='d(t_pure)', by="min_cover_size", showfliers=False)
#     #data.groupby('min_cover_size')
#     #data[title].plot.kde()
#     # factor = data_cut['min_cover_size'].nunique() // 15 + 1
#     # data_cut['min_cover_size'] = data_cut['min_cover_size'].astype(int) // factor * factor

#     # sns.violinplot(data_cut, x='min_cover_size', y=tcol, inner=None).set(title=title)
#     plt.title(title)
#     plt.ylabel('Query Latency (ns)')
#     plt.xlabel('Cover size')
#     c = data['min_cover_size'].unique()

#     plt.xticks(range(min(c), max(c) + (max(c) - min(c)) // 20 + 1, (max(c) - min(c)) // 20 + 1))
#     plt.show()


    
# # Fix UB=511, C=11
# data_fix_ub_cov = all_data.query('ub == 511 and min_cover_size == 11')
# data_fix_ub_cov['d(t_pure)'] /= 1e6


# p = data_fix_ub_cov.groupby(["sp", "cf"]).mean().unstack(level=0).plot(y='d(t_pure)', marker='o')
# plt.ylim(bottom=0)
# plt.xlabel('Contention factor')
# plt.xticks([0, 4, 8])
# plt.ylabel('Average query latency (ms)')
# plt.title('Average query latency vs contention factor, UB=511, cover size=11')
# plt.legend(title='Sparsity')
# plt.plot()

# p = data_fix_ub_cov.groupby(["cf", "sp"]).mean().unstack(level=0).plot(y='d(t_pure)', marker='o')
# plt.ylim(bottom=0)
# plt.xlabel('Sparsity')
# plt.xticks([1, 2, 3, 4])
# plt.ylabel('Average query latency (ms)')
# plt.title('Average query latency vs sparsity, UB=511, cover size=11')
# plt.legend(title='Contention factor')
# plt.plot()

# # Fix UB=511, cf=0
# data_fix_ub_cf = all_data.query('ub == 511 and cf == 0')
# data_fix_ub_cf['d(t_pure)'] /= 1e6
# data_fix_ub_cf
# #q = data_fix_ub_cf['d(t_pure)'].quantile(0.999)
# #with pd.option_context('display.max_rows', None, 'display.max_columns', None): 
# #    print(data_fix_ub_cf[data_fix_ub_cf['d(t_pure)'] > q])
# for sp in [1, 2, 3, 4]:
# # for ub in [511, 8191, 131071, 2**31-1]:
# #      for cf in [0,4,8]:
#      all_data[(all_data['sp'] == sp)].groupby(['min_cover_size']).quantile(0.99).plot(y='d(t_pure)')
#      plt.title(f'99th percentile latency vs cover size, SP={sp}')
#      plt.xlabel('Cover size')
#      plt.ylabel('Query latency (ns)')
#      plt.axvline(28, color='lightgray', linestyle='--')
#      plt.savefig(f'figures/latency99_cov_ub{ub}_cf{cf}.png')

# #all_data[all_data['ub'] == 8191].groupby(['min_cover_size']).quantile(0.99)['d(t_pure)'].plot(y='d(t_pure)')
# #all_data[all_data['ub'] == 131071].groupby(['min_cover_size']).quantile(0.99)['d(t_pure)'].plot(y='d(t_pure)')
# #all_data[all_data['ub'] == 2**31-1].groupby(['min_cover_size']).quantile(0.99)['d(t_pure)'].plot(y='d(t_pure)')
# #plt.xlim(25, 35)
# #plt.ylim(1e7, 2e7)
# weirdo = data_fix_ub_cf[data_fix_ub_cf['min_cover_size'] == 28]
# not_weirdo = data_fix_ub_cf[data_fix_ub_cf['min_cover_size'] == 25]
# a = weirdo.hist(column='d(t_pure)', bins=1000)
# plt.xlim(left=0)
# b = not_weirdo.hist(column='d(t_pure)', bins=1000)
# plt.xlim(left=0)
# for q in [0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 0.995, 0.998, 0.999]:
#      print(q, weirdo['d(t_pure)'].quantile(q) / weirdo['d(t_pure)'].mean(), not_weirdo['d(t_pure)'].quantile(q) / not_weirdo['d(t_pure)'].mean())
# for cs in all_data['min_cover_size'].unique():
#      print(cs, len(all_data[all_data['min_cover_size'] == cs]))
# #data_cut.groupby(['min_cover_size']).mean()['d(t_pure)'].plot(y='d(t_pure)')
# #data_fix_ub_cf.groupby(['min_cover_size']).mean()['d(t_pure)'].plot(y='d(t_pure)')
# #for cs in data_cut['min_cover_size'].unique():
# #     print(cs, len(data_cut[data_cut['min_cover_size'] == cs]))
# def func(x, a):
#     return a * (x - 1) + mean_by_cov[1]

# mean_by_cov = data_fix_ub_cf.groupby(['min_cover_size']).mean()['d(t_pure)']
# #plt.plot(mean_by_cov.index, np.poly1d(np.polyfit(mean_by_cov.index, mean_by_cov, 1))(mean_by_cov.index))
# print(mean_by_cov.index)
# mean_by_cov.plot(y='d(t_pure)', label='Actual')
# popt, pcov = curve_fit(func, mean_by_cov.index, mean_by_cov)
# print(popt)
# print([0] + list(mean_by_cov.index))
# plt.plot(mean_by_cov.index, func(mean_by_cov.index, popt),"r--", label='Best fit')
# plt.ylim(bottom=0)
# plt.xlim(left=0)
# plt.xlabel('Query cover size')
# plt.ylabel('Average query latency (ms)')
# plt.title('Average query latency vs cover size, UB=511, contention=0')
# plt.legend()
# plt.plot()
# #for cs in data_cut['min_cover_size'].unique():
# #     print(cs, len(data_cut[data_cut['min_cover_size'] == cs]))
# def func(x, a):
#     return a * (x - 1) + mean_by_cov[1]

# q = data_fix_ub_cf['d(t_pure)'].quantile(0.999)
# data_cut = data_fix_ub_cf[data_fix_ub_cf['d(t_pure)'] < q]
# mean_by_cov = data_cut.groupby(['min_cover_size']).mean()['d(t_pure)']
# #plt.plot(mean_by_cov.index, np.poly1d(np.polyfit(mean_by_cov.index, mean_by_cov, 1))(mean_by_cov.index))
# print(mean_by_cov.index)
# mean_by_cov.plot(y='d(t_pure)', label='Actual')
# popt, pcov = curve_fit(func, mean_by_cov.index, mean_by_cov)
# print(popt)
# print([0] + list(mean_by_cov.index))
# plt.plot(mean_by_cov.index, func(mean_by_cov.index, popt),"r--", label="Best fit")
# plt.ylim(bottom=0)
# plt.xlim(left=0)
# plt.xlabel('Query cover size')
# plt.ylabel('Average query latency (ms)')
# plt.title('Average query latency vs cover size, UB=511, contention=0 (< 99.9 percentile)')
# plt.legend()
# plt.plot()
# #for cs in data_cut['min_cover_size'].unique():
# #     print(cs, len(data_cut[data_cut['min_cover_size'] == cs]))